In [ ]:
Inputs. Hidden. Output.

In [2]:
import torch
import torch.nn.functional as F
import random
torch.set_printoptions(sci_mode=False)

words = open("names.txt", "r").read().splitlines()
random.shuffle(words)

d_h = 10
d_emb = 10
d_V = 27


alphabet = list("abcdefghijklmnopqrstuvwxyz")
stoi = {alphabet[i] : i + 1 for i in range(26)}
stoi["."] = 0
itos = {i : letter for letter, i in stoi.items()}
vocab_size = len(alphabet) + 1

In [30]:
initial_hidden = torch.randn(d_h, 1)*0.01
U = torch.randn(d_h, d_h)
W = torch.randn(d_h, d_emb)
V = torch.randn(d_V, d_h)*0.01

b_h = torch.randn(d_h, 1)*0.0001
b_y = torch.randn(d_V, 1)*0.0001

emb = torch.randn(d_emb, d_V)*0.01

params = [U, W, V, emb, initial_hidden, b_h, b_y]

for p in params:
    p.requires_grad = True

In [31]:
def one_hot(X, num_classes=None):
    length = X+1 if num_classes is None else num_classes
    out = torch.zeros(length, 1)
    out[X] = 1
    return out

def cross_entropy(output, target):
    output_max = torch.max(output)
    output = output - output_max
    output_exp = torch.exp(output)
    output_exp_sum = output_exp.sum()
    output = output_exp/output_exp_sum
    out = (output*target).sum()
    out = -torch.log(out)
    return out

In [32]:
kwargs = {
    "lr_arch" : "adam",
    "beta1": 0.9,
    "beta2": 0.9,
    "eps" : 1e-6,
    "gamma" : 0.9,
    "momentum" : 0.1,
    "learn" : 0.001,
    "batch_size" : 16,
    "iterations" : len(words)
}

In [33]:
def train_model(lr_arch = None, momentum = 0.1, beta1 = 0.9, beta2 = 0.9 ,gamma = 0.9, eps = 1e-5, learn = 0.1, batch_size = 32, iterations = 1000):

    if lr_arch == "momentum based":
        velocities = [torch.zeros_like(p) for p in params]
    if lr_arch == "adagrad":
        accum_grad = [torch.zeros_like(p) for p in params]
    if lr_arch == "rmsprop":
        accum_grad_exp = [torch.zeros_like(p) for p in params]
    if lr_arch ==  "adam":
        accum_mt = [torch.zeros_like(p) for p in params]
        accum_vt = [torch.zeros_like(p) for p in params]

    for j in range(iterations):
        ix = torch.randint(0, len(words), (batch_size,))
        # ix = [j]
        word = ""
        for x in ix:
            word += "." + words[x]
        hidden_layer = initial_hidden
        losses = []
        word = word + "."
        X = word[:-1]
        Y = word[1:]

        for i in range(len(X)):
            letter = X[i]
            idx = stoi[letter]
            e = emb @ one_hot(idx, num_classes=d_V)
            hidden_layer = torch.tanh(W @ e + U @ hidden_layer + b_h)
            y = V @ hidden_layer + b_y
            losses.append(cross_entropy(y, one_hot(stoi[Y[i]], num_classes=27)))

        for p in params:
            p.grad = None

        loss = torch.stack(losses).mean()
        loss.backward()
        if j % 1000 == 0:
            print(f"{j}/{iterations} | {loss}")

        lr = learn
        for i in range(len(params)):
            if lr_arch == "adam":
                accum_mt[i] = beta1*accum_mt[i] + (1-beta1)*params[i].grad
                accum_vt[i] = beta2*accum_vt[i] + (1-beta2)*params[i].grad**2
                t = j + 1
                mt_hat = accum_mt[i]/(1-beta1**t)
                vt_hat = accum_vt[i]/(1-beta2**t)
                update = (mt_hat/(vt_hat**0.5 + eps)) * lr

            elif lr_arch == "rmsprop":
                accum_grad_exp[i] = gamma*accum_grad_exp[i] + (1-gamma)*params[i].grad**2
                update = (params[i].grad/(accum_grad_exp[i] + eps)**0.5) * lr

            elif lr_arch == "momentum based":
                velocities[i] = lr*params[i].grad + momentum*velocities[i]
                update = velocities[i]

            elif lr_arch == "adagrad":
                accum_grad[i] = accum_grad[i] + params[i].grad**2
                update = (params[i].grad/(accum_grad[i] + eps)**0.5) * lr
            else:
                update = lr*params[i].grad

            params[i].data += -update

In [34]:
train_model(**kwargs)

0/32033 | 3.290806531906128
1000/32033 | 2.534318447113037
2000/32033 | 2.5941390991210938
3000/32033 | 2.585143566131592
4000/32033 | 2.4750871658325195
5000/32033 | 2.5267021656036377
6000/32033 | 2.388293981552124
7000/32033 | 2.4036312103271484
8000/32033 | 2.4697391986846924
9000/32033 | 2.4659993648529053
10000/32033 | 2.439314365386963
11000/32033 | 2.3522119522094727
12000/32033 | 2.41939377784729
13000/32033 | 2.328529119491577
14000/32033 | 2.462815284729004
15000/32033 | 2.342440128326416
16000/32033 | 2.4086642265319824
17000/32033 | 2.414179563522339
18000/32033 | 2.3509364128112793
19000/32033 | 2.324404239654541
20000/32033 | 2.3551838397979736
21000/32033 | 2.3889992237091064
22000/32033 | 2.275195837020874
23000/32033 | 2.334052085876465
24000/32033 | 2.386460542678833
25000/32033 | 2.272052049636841
26000/32033 | 2.4081637859344482
27000/32033 | 2.3560738563537598
28000/32033 | 2.259721040725708
29000/32033 | 2.3856053352355957
30000/32033 | 2.4244415760040283
31000/3

In [43]:
y = 0
set_words = set(words)
i = 0
num = 10
for _ in range(num):
    hidden_layer = initial_hidden
    word = ""
    while True:
        e = emb @ one_hot(y, num_classes=d_V)
        hidden_layer = torch.tanh(W @ e + U @ hidden_layer + b_h)
        y = V @ hidden_layer + b_y
        y = torch.multinomial(torch.softmax(y, dim = 0).view(-1), num_samples=1).item()
        word += itos[y]
        if itos[y] == ".":
            if word not in set_words:
                print(word)
            if word in set_words:
                i += 1

            break
print(i/num*100)

ramyya.
emaama.
laamon.
kreci.
leniin.
savo.
disyin.
yyleu.
kehdan.
josyon.
0.0


In [41]:
hidden_layer = initial_hidden
e = emb @ one_hot(0, num_classes=d_V)
hidden_layer = torch.tanh(W @ e + U @ hidden_layer)
y = V @ hidden_layer
probs = torch.softmax(y, dim=0).view(-1)
print(probs)
print(itos[probs.argmax().item()], probs.max().item())

tensor([0.1614, 0.0294, 0.0350, 0.0502, 0.0300, 0.0189, 0.0424, 0.0763, 0.0071,
        0.0046, 0.0999, 0.0515, 0.0210, 0.0297, 0.1019, 0.0136, 0.0309, 0.0184,
        0.0179, 0.0404, 0.0209, 0.0041, 0.0122, 0.0240, 0.0185, 0.0074, 0.0325],
       grad_fn=<ViewBackward0>)
. 0.1614062339067459
